# Lax-generated radially polarized beam

This notebook is the radial-polarization counterpart of `lax_gaussian_corrections_visualization.ipynb`.

It keeps the same final three-plane visualization, but changes the paraxial seed to a **longitudinal vector potential**

\[
\mathbf A^{(0)}=(0,0,A_{\rm amp}\psi_0).
\]

For this construction the magnetic field is azimuthal. The phrase *radially polarized beam* refers to the electric-field polarization; the magnetic field visualized here is transverse and azimuthal.

The notebook is intentionally limited to the magnetic field and to the same external long temporal gate used in the previous visualization.

> **Boundary-convention note.** This notebook uses the generic Lax generator in `lax_series.py`, which fixes the higher-order homogeneous freedom by requiring
> \[
> A^{(2j)}(X,Y,0)=0,\qquad j>0.
> \]
> Therefore this is not the specific higher-order Salamin boundary convention, although the magnetic-field parity is the same:
> \[
> \mathbf B=\epsilon\mathbf B^{(1)}
> +\epsilon^3\mathbf B^{(3)}
> +\epsilon^5\mathbf B^{(5)}+\cdots.
> \]

## 1. Imports and physical parameters

The visualization coordinates are \((x_0,x_1,x_2)\), with propagation along \(x_0\).

The Lax backend uses \((X,Y,Z)\), with propagation along \(Z\):

\[
X=\frac{x_1}{w_0},\qquad
Y=\frac{x_2}{w_0},\qquad
Z=\frac{x_0}{z_R}.
\]

With

\[
\epsilon=\frac{2}{kw_0},\qquad
z_R=\frac{k w_0^2}{2},
\]

the physical transverse derivatives are

\[
\partial_{x_1}=\frac{k\epsilon}{2}\partial_X,\qquad
\partial_{x_2}=\frac{k\epsilon}{2}\partial_Y.
\]

The temporal Gaussian is applied **after** constructing the monochromatic Lax field, so it acts only as a visualization gate.

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

from lax_series import (
    lax_expand_vector,
    paraxial_residual,
    compile_vector_field,
)

sp.init_printing()

# Physical parameters, aligned with the previous visualization.
l0 = 2.0 * np.pi
k0 = 1.0
omega = 1.0
waist = l0

# Longitudinal vector-potential amplitude.
Aamp0 = 1.0

zR = k0 * waist**2 / 2.0
eps0 = 2.0 / (k0 * waist)

# Natural normalization for this longitudinal-potential construction.
Bref = k0 * Aamp0

Lsim = [5.0 * l0, 4.0 * l0, 2.5 * l0]
t0 = l0
Tsim = 18.0 * t0

# Deliberately long external visualization envelope.
fwhm = 6.0 * t0

def time_envelope(t):
    sigma = (0.5 * fwhm)**2 / np.log(2.0)
    return np.exp(-(t**2) / sigma)

print(f"epsilon = {eps0:.6f}")
print(f"zR      = {zR:.6f}")
print(f"Bref=k*Aamp = {Bref:.6f}")

## 2. Longitudinal Gaussian seed and Lax expansion

For the carrier convention used by `lax_series.py`,

\[
e^{ikx_0-i\omega t},
\]

use

\[
f(Z)=\frac{1}{1+iZ},
\qquad
\psi_0(X,Y,Z)
=
f\,e^{-f(X^2+Y^2)}.
\]

The only beam-specific input is

\[
\boxed{
\mathbf A^{(0)}
=
(0,0,A_{\rm amp}\psi_0)
}.
\]

The generic Lax generator then constructs

\[
A_Z
=
A_Z^{(0)}
+\epsilon^2 A_Z^{(2)}
+\epsilon^4 A_Z^{(4)}
+O(\epsilon^6).
\]

In [ ]:
I = sp.I

X, Y, Z = sp.symbols("X Y Z", real=True)
eps, k, Aamp = sp.symbols("eps k Aamp", positive=True, real=True)

rho2 = X**2 + Y**2
f = 1 / (1 + I * Z)
psi0 = f * sp.exp(-f * rho2)

A0 = (
    sp.Integer(0),
    sp.Integer(0),
    Aamp * psi0,
)

seed_residual = sp.simplify(paraxial_residual(psi0, (X, Y, Z)))
print("paraxial residual of psi0:", seed_residual)

# Generate A^(0) + eps^2 A^(2) + eps^4 A^(4).
A_lax = lax_expand_vector(
    A0,
    coords=(X, Y, Z),
    eps=eps,
    order=4,
)

Az_expanded = sp.expand(A_lax[2])
Az_coeff = {
    n: Az_expanded.coeff(eps, n)
    for n in (0, 2, 4)
}

print("generated A orders:", sorted(Az_coeff))

## 3. Magnetic-field orders

For a purely longitudinal vector potential,

\[
\mathbf A=A_Z\hat{\mathbf Z},
\]

the carrier contribution \(ik\hat{\mathbf Z}\times\mathbf A\) vanishes identically. Hence

\[
\mathbf B=\nabla\times\mathbf A
\]

contains only transverse derivatives:

\[
B_X=\frac{k\epsilon}{2}\,\partial_Y A_Z,
\qquad
B_Y=-\frac{k\epsilon}{2}\,\partial_X A_Z,
\qquad
B_Z=0.
\]

Therefore an even Lax series for \(A_Z\) produces only odd orders in \(B\):

\[
\boxed{
\mathbf B
=
\epsilon\mathbf B^{(1)}
+\epsilon^3\mathbf B^{(3)}
+\epsilon^5\mathbf B^{(5)}
+\cdots
}.
\]

The code below constructs these order contributions directly from the Lax-generated coefficients of \(A_Z\). This is algebraically the same curl used by the generic backend, but evaluating it order-by-order keeps the interactive notebook fast.

On the positive \(X\) axis, \(B_\theta=B_Y\), and the leading term is

\[
\frac{B_\theta^{[1]}}{kA_{\rm amp}}
=
\epsilon X f^2 e^{-fX^2}.
\]

In [ ]:
zero = sp.Integer(0)

# Coefficients B^(n), before multiplying by eps**n.
# Backend component order: (B_X, B_Y, B_Z).
B_coeff = {
    1: (
        k * sp.diff(Az_coeff[0], Y) / 2,
        -k * sp.diff(Az_coeff[0], X) / 2,
        zero,
    ),
    3: (
        k * sp.diff(Az_coeff[2], Y) / 2,
        -k * sp.diff(Az_coeff[2], X) / 2,
        zero,
    ),
    5: (
        k * sp.diff(Az_coeff[4], Y) / 2,
        -k * sp.diff(Az_coeff[4], X) / 2,
        zero,
    ),
}

# Actual order contributions, including eps**n.
B_order_expr = {
    n: tuple(eps**n * component for component in B_coeff[n])
    for n in (1, 3, 5)
}

B_cumulative_expr = {
    1: B_order_expr[1],
    3: tuple(B_order_expr[1][j] + B_order_expr[3][j] for j in range(3)),
    5: tuple(
        B_order_expr[1][j]
        + B_order_expr[3][j]
        + B_order_expr[5][j]
        for j in range(3)
    ),
}

# Symbolic check of the leading azimuthal field.
R = sp.symbols("R", nonnegative=True, real=True)
leading_btheta = sp.simplify(
    B_order_expr[1][1].subs({X: R, Y: 0}) / (k * Aamp)
)
expected_leading_btheta = eps * R * f**2 * sp.exp(-f * R**2)

print(
    "leading B_theta identity:",
    sp.simplify(leading_btheta - expected_leading_btheta) == 0,
)

# The generic generator preserves the supplied focal profile, so its
# generated higher-order corrections vanish at Z=0.
for n in (3, 5):
    focus_zero = all(
        sp.simplify(component.subs(Z, 0)) == 0
        for component in B_order_expr[n]
    )
    print(f"order {n} correction vanishes at Z=0:", focus_zero)

# Fix physical parameters before lambdifying.
parameter_subs = {
    eps: eps0,
    k: k0,
    Aamp: Aamp0,
}

B_order_fn = {
    n: compile_vector_field(
        tuple(component.subs(parameter_subs) for component in B_order_expr[n]),
        (X, Y, Z),
    )
    for n in (1, 3, 5)
}

B_cumulative_fn = {
    n: compile_vector_field(
        tuple(component.subs(parameter_subs) for component in B_cumulative_expr[n]),
        (X, Y, Z),
    )
    for n in (1, 3, 5)
}

print("compiled B orders:", [1, 3, 5])

## 4. Numerical field adapter

`Bfield_meshed` is the only field interface used by the visualization.

It converts local coordinates to \((X,Y,Z)\), evaluates the chosen Lax view, multiplies by the carrier \(e^{ikx_0-i\omega t}\), maps the vector back to the local \((x_0,x_1,x_2)\) basis, and finally applies the long temporal visualization envelope.

The component mapping is

\[
(B_X,B_Y,B_Z)
\longrightarrow
(B_{x_0},B_{x_1},B_{x_2})
=
(B_Z,B_X,B_Y).
\]

For the unrotated beam,

\[
\boxed{B_{x_0}=0}
\]

at every retained order. The transverse pair \((B_{x_1},B_{x_2})\) forms the azimuthal magnetic field.

In [ ]:
def _vectorize_compiled_result(raw, Xn, Yn, Zn):
    """Broadcast symbolic scalar zeros to the mesh shape."""
    shape = np.broadcast(
        np.asarray(Xn),
        np.asarray(Yn),
        np.asarray(Zn),
    ).shape

    return np.stack([
        np.broadcast_to(np.asarray(component, dtype=complex), shape)
        for component in raw
    ], axis=0)


def _evaluate_backend_envelope(Xn, Yn, Zn, order, display_mode):
    if order not in (1, 3, 5):
        raise ValueError("For this radial construction, order must be 1, 3, or 5.")

    if display_mode == "cumulative":
        raw = B_cumulative_fn[order](Xn, Yn, Zn)
    elif display_mode == "correction":
        raw = B_order_fn[order](Xn, Yn, Zn)
    else:
        raise ValueError("display_mode must be 'cumulative' or 'correction'.")

    return _vectorize_compiled_result(raw, Xn, Yn, Zn)


def Bfield_meshed(x, t, order=1, display_mode="cumulative", apply_pulse=True):
    """Return the real local magnetic field with shape (3, ...)."""
    x = np.asarray(x)

    Xn = x[1] / waist
    Yn = x[2] / waist
    Zn = x[0] / zR

    B_backend = _evaluate_backend_envelope(
        Xn, Yn, Zn, order, display_mode
    )

    # Backend (X,Y,Z) -> local (x1,x2,x0).
    B_local_envelope = np.stack([
        B_backend[2],  # local x0 = longitudinal B
        B_backend[0],  # local x1
        B_backend[1],  # local x2
    ], axis=0)

    carrier = np.exp(1j * (k0 * x[0] - omega * t))
    B_real = np.real(B_local_envelope * carrier)

    if apply_pulse:
        B_real = B_real * time_envelope(t - x[0])

    return B_real


# Small numerical symmetry check in the unrotated local frame.
test_axis = np.linspace(-2.0 * waist, 2.0 * waist, 61)
T1, T2 = np.meshgrid(test_axis, test_axis)
test_coords = np.asarray([
    np.zeros_like(T1),
    T1,
    T2,
])

B_test = Bfield_meshed(
    test_coords,
    t=0.0,
    order=5,
    display_mode="cumulative",
)

print(
    "max longitudinal |B_x0| / Bref:",
    np.max(np.abs(B_test[0])) / Bref,
)

## 5. Rotation and offset

The transformation layer is unchanged from the previous notebook.

The local radial beam has \(B_{x_0}=0\), but after rotating the beam a selected **laboratory** Cartesian component can contain contributions from both transverse local components.

To inspect the intrinsic azimuthal structure first, use

\[
\theta=\phi=0.
\]

In [ ]:
def RotM(axis, angle):
    """Rotation matrices in the convention used by transform_test2.ipynb."""
    if axis == 'x':
        return np.asarray([
            [1, 0, 0],
            [0, np.cos(angle), -np.sin(angle)],
            [0, np.sin(angle),  np.cos(angle)],
        ])
    if axis == 'y':
        return np.asarray([
            [ np.cos(angle), 0, np.sin(angle)],
            [0, 1, 0],
            [-np.sin(angle), 0, np.cos(angle)],
        ])
    if axis == 'z':
        return np.asarray([
            [np.cos(angle), -np.sin(angle), 0],
            [np.sin(angle),  np.cos(angle), 0],
            [0, 0, 1],
        ])
    raise ValueError("axis must be 'x', 'y', or 'z'.")


def transform_vector_field_vectorised(R, offset, vector, selection=(0, 1, 2)):
    def vector_field_transformed(x_, t_):
        off = np.asarray(offset).reshape((3,) + (1,) * (x_.ndim - 1))

        x_rot = np.einsum('ij,j...->i...', R, x_ - off)
        vec = vector(x_rot, t_)
        vec_rot = np.einsum('ij,j...->i...', R.T, vec)

        return np.asarray([vec_rot[idx] for idx in selection])

    return vector_field_transformed


def transformed_selected_component(
    coords,
    t,
    theta,
    phi,
    offset,
    selected_component,
    order,
    display_mode,
):
    R = RotM('z', theta) @ RotM('y', phi)

    def selected_lax_field(local_x, local_t):
        return Bfield_meshed(
            local_x,
            local_t,
            order=order,
            display_mode=display_mode,
            apply_pulse=True,
        )

    transformed = transform_vector_field_vectorised(
        R,
        np.asarray(offset),
        selected_lax_field,
        selection=(selected_component,),
    )

    return np.squeeze(transformed(coords, t))

## 6. Three-plane visualization

The output is the same three orthogonal slices as in the linear-Gaussian notebook:

- \(x_0\)-\(x_1\) at fixed \(x_2\);
- \(x_0\)-\(x_2\) at fixed \(x_1\);
- \(x_1\)-\(x_2\) at fixed \(x_0\).

The Lax-order control contains only the physically present magnetic-field orders:

- **1**: \(O(\epsilon)\);
- **3**: through \(O(\epsilon^3)\);
- **5**: through \(O(\epsilon^5)\).

The `view` control distinguishes:

- **cumulative**: all retained odd orders through the selected order;
- **isolated correction**: only the selected \(\epsilon^n\mathbf B^{(n)}\) contribution.

The plotted normalization is

\[
B_{\rm ref}=kA_{\rm amp}.
\]

In [ ]:
N = 250
cmap = "RdBu_r"

x0_vals = np.linspace(-Lsim[0], Lsim[0], N)
x1_vals = np.linspace(-Lsim[1], Lsim[1], N)
x2_vals = np.linspace(-Lsim[2], Lsim[2], N)

X01_0, X01_1 = np.meshgrid(x0_vals, x1_vals)
X02_0, X02_2 = np.meshgrid(x0_vals, x2_vals)
X12_1, X12_2 = np.meshgrid(x1_vals, x2_vals)


def _safe_limit(field):
    limit = float(np.max(np.abs(field)))
    return limit if limit > 0.0 else 1.0


def plot_field_3planes(
    t,
    x0,
    x1,
    x2,
    theta,
    phi,
    off0,
    off1,
    off2,
    selected_component,
    order,
    display_mode,
):
    offset = (off0, off1, off2)

    coords01 = np.asarray([
        X01_0,
        X01_1,
        np.full_like(X01_0, x2),
    ])
    coords02 = np.asarray([
        X02_0,
        np.full_like(X02_0, x1),
        X02_2,
    ])
    coords12 = np.asarray([
        np.full_like(X12_1, x0),
        X12_1,
        X12_2,
    ])

    field01 = transformed_selected_component(
        coords01, t, theta, phi, offset,
        selected_component, order, display_mode,
    ) / Bref
    field02 = transformed_selected_component(
        coords02, t, theta, phi, offset,
        selected_component, order, display_mode,
    ) / Bref
    field12 = transformed_selected_component(
        coords12, t, theta, phi, offset,
        selected_component, order, display_mode,
    ) / Bref

    limits = [_safe_limit(field01), _safe_limit(field02), _safe_limit(field12)]
    maxima = [
        float(np.max(np.abs(field01))),
        float(np.max(np.abs(field02))),
        float(np.max(np.abs(field12))),
    ]

    fig, axs = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)

    component_names = {
        0: 'x0 (longitudinal)',
        1: 'x1',
        2: 'x2',
    }
    component = component_names[selected_component]

    if display_mode == "cumulative":
        order_label = f"through O(eps^{order})"
    else:
        order_label = f"isolated O(eps^{order})"

    panels = [
        (
            axs[0], field01,
            [-Lsim[0], Lsim[0], -Lsim[1], Lsim[1]],
            'x0', 'x1', f'x0-x1, x2={x2:.3f}',
            limits[0], maxima[0],
        ),
        (
            axs[1], field02,
            [-Lsim[0], Lsim[0], -Lsim[2], Lsim[2]],
            'x0', 'x2', f'x0-x2, x1={x1:.3f}',
            limits[1], maxima[1],
        ),
        (
            axs[2], field12,
            [-Lsim[1], Lsim[1], -Lsim[2], Lsim[2]],
            'x1', 'x2', f'x1-x2, x0={x0:.3f}',
            limits[2], maxima[2],
        ),
    ]

    for ax, field, extent, xlabel, ylabel, plane_title, limit, maximum in panels:
        im = ax.imshow(
            field,
            extent=extent,
            origin='lower',
            aspect='auto',
            vmin=-limit,
            vmax=limit,
            cmap=cmap,
        )

        ax.set_title(
            f'{plane_title}\n'
            f'B_{component}, {order_label}; '
            f'max |B/Bref|={maximum:.3e}'
        )
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)

        cbar = fig.colorbar(im, ax=ax, orientation='horizontal', pad=0.12)
        cbar.set_label('B / Bref')

    plt.show()

In [ ]:
from ipywidgets import interact, FloatSlider, Dropdown

interact(
    plot_field_3planes,

    t=FloatSlider(
        min=-Tsim, max=Tsim, step=Tsim / 200,
        value=0.0, description='t',
    ),
    x0=FloatSlider(
        min=-Lsim[0], max=Lsim[0], step=(2 * Lsim[0]) / 200,
        value=0.0, description='x0',
    ),
    x1=FloatSlider(
        min=-Lsim[1], max=Lsim[1], step=(2 * Lsim[1]) / 200,
        value=0.0, description='x1',
    ),
    x2=FloatSlider(
        min=-Lsim[2], max=Lsim[2], step=(2 * Lsim[2]) / 200,
        value=0.0, description='x2',
    ),
    theta=FloatSlider(
        min=-0.5 * np.pi, max=0.5 * np.pi, step=np.pi / 200,
        value=0.0, description='theta',
    ),
    phi=FloatSlider(
        min=-0.5 * np.pi, max=0.5 * np.pi, step=np.pi / 200,
        value=0.0, description='phi',
    ),
    off0=FloatSlider(
        min=-Lsim[0], max=Lsim[0], step=(2 * Lsim[0]) / 200,
        value=0.0, description='off0',
    ),
    off1=FloatSlider(
        min=-Lsim[1], max=Lsim[1], step=(2 * Lsim[1]) / 200,
        value=0.0, description='off1',
    ),
    off2=FloatSlider(
        min=-Lsim[2], max=Lsim[2], step=(2 * Lsim[2]) / 200,
        value=0.0, description='off2',
    ),
    selected_component=Dropdown(
        options=[
            ('B_x0 (longitudinal)', 0),
            ('B_x1', 1),
            ('B_x2', 2),
        ],
        value=2,
        description='component',
    ),
    order=Dropdown(
        options=[
            ('1  : O(eps)', 1),
            ('3  : through O(eps^3)', 3),
            ('5  : through O(eps^5)', 5),
        ],
        value=1,
        description='B order',
    ),
    display_mode=Dropdown(
        options=[
            ('cumulative', 'cumulative'),
            ('isolated correction', 'correction'),
        ],
        value='cumulative',
        description='view',
    ),
);